In [4]:
!pip install -q boto3 pandas pyarrow wandb tf2onnx onnx tinygrad drift-sdk av openpilot-logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 142.9 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
"""
Train a driving policy from data downloaded through the drift SDK.
"""

from __future__ import annotations

import argparse
import base64
import csv
import importlib
import json
import logging
import math
import os
import shutil
import tempfile
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import drift
import numpy as np
import pandas as pd
import tensorflow as tf


# =============================================================================
# Configuration
# =============================================================================


def writable_dir(path: str | os.PathLike[str], fallback: str | os.PathLike[str]) -> Path:
    path = Path(path)
    try:
        path.mkdir(parents=True, exist_ok=True)
        return path
    except OSError:
        fallback = Path(fallback)
        fallback.mkdir(parents=True, exist_ok=True)
        return fallback


DATA_DIR = Path("/content/irl_recordings")
OUTPUT_DIR = writable_dir("/content/drift_irl_policy_one_pedal_v1", "/tmp/drift_irl_policy_one_pedal_v1")

EPOCHS = 1000
LATENT_ENCODING_BATCH_SIZE = 1000
TRAIN_BATCH_SIZE = 15
LEARNING_RATE = 3e-6
VALIDATION_FRACTION = 0.20
SAMPLE_HZ = 5.0
SOURCE_VIDEO_FPS = 20.0
STACK_SIZE = 51
ACTIVE_STACK_SIZE = STACK_SIZE
SPLIT_BLOCK_SECONDS = 20.0
NAV_MAX_STEPS = 16
STEERING_STRAIGHT_THRESHOLD = 0.02
PEDAL_ACTIVE_THRESHOLD = 0.02
SEED = 42
RESUME_TRAINING = True
ROLLOUT_LEN = 10
ROLLOUT_LOG_EVERY_EPOCHS = 5
ROLLOUT_PREVIEW_COUNT = 16
RESUME_EPOCH = 50
IMAGE_HEIGHT = 96
IMAGE_WIDTH = 160
VAE_DOWNSAMPLE_FACTOR = 16
LATENT_GRID_HEIGHT = IMAGE_HEIGHT // VAE_DOWNSAMPLE_FACTOR
LATENT_GRID_WIDTH = IMAGE_WIDTH // VAE_DOWNSAMPLE_FACTOR
LATENT_CHANNELS = 64
THROTTLE_LOSS_WEIGHT = 1.0
BRAKE_LOSS_WEIGHT = 1.0
STEERING_LOSS_WEIGHT = 1.0
VAE_LATENT_DIM = 128
TRAIN_RGB_ENCODER_E2E = False
RGB_ENCODER_SOURCE = "vae"
RGB_ENCODER_TRAINABLE = False
EXPORT_TRAINED_RGB_ENCODER = False
FUSION_LATENT_DIM = 128
FUSION_LATENT_KL_LOSS_WEIGHT = 1e-4
CONTROL_HISTORY_DIM = 4
STEERING_ANGLE_SCALE_DEG = 540.0
CAN_BUS_POWERTRAIN = 0
CAN_ACCELERATOR_PEDAL = 0x1C4
CAN_BRAKE_PEDAL = 0x0BE
CAN_STEERING_ANGLE = 0x1E5
CAN_SIGNAL_MAX_AGE_SECONDS = 0.05
CONTROL_TARGET_DELAY_SECONDS = 0.20
CONTROL_TARGET_MAX_ERROR_SECONDS = 0.04
CONTROL_TIMELINE_MAX_GAP_SECONDS = 0.12
DATASET_PERCENT = 100.0
CHECKPOINT_EVERY_EPOCHS = 50
EXPORT_POLICY_ONNX = True
POLICY_ONNX_BATCH_SIZE = 1
POLICY_ONNX_OPSET = 17
VERIFY_POLICY_ONNX_TINYGRAD = True
POLICY_ONNX_DISALLOWED_OPS = {"Loop", "If", "Scan"}
POLICY_ONNX_OUTPUT_NAMES = ("throttle_magnitude", "brake_magnitude", "steering")
POLICY_ONNX_FAST_RANK3_DENSE = True
POLICY_ONNX_SINGLE_OUTPUT = True
POLICY_ONNX_SINGLE_OUTPUT_NAME = "policy_output"
EXPORT_VISION_ONNX = True
VISION_ONNX_BATCH_SIZE = 1
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "training.log"
CHECKPOINT_PATTERN = "policy_epoch_{epoch:04d}.keras"
HISTORY_PATH = OUTPUT_DIR / "epoch_metrics.csv"
LOGGER = logging.getLogger("driving_policy")
LOGGER.setLevel(logging.DEBUG)
LOGGER.handlers.clear()
LOGGER.propagate = False
LOG_FORMAT = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
console_handler = logging.StreamHandler()
console_handler.setLevel("WARNING")
console_handler.setFormatter(LOG_FORMAT)
file_handler = logging.FileHandler(LOG_PATH, mode="a" if RESUME_TRAINING else "w")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(LOG_FORMAT)
LOGGER.addHandler(console_handler)
LOGGER.addHandler(file_handler)


# =============================================================================
# Generic Helpers
# =============================================================================

def import_module_or_raise(module_name: str, package_name: Optional[str] = None) -> Any:
    try:
        return importlib.import_module(module_name)
    except ImportError as exc:
        package_hint = package_name or module_name
        raise RuntimeError(f"Missing dependency '{module_name}'. Install it with: !pip install {package_hint}") from exc


def import_attr_or_raise(module_name: str, attr_name: str, package_name: Optional[str] = None) -> Any:
    module = import_module_or_raise(module_name, package_name=package_name)
    return getattr(module, attr_name)


def console_status(message: str, *args: Any) -> None:
    text = message % args if args else message
    LOGGER.info(text)
    print(text, flush=True)


def log_section(title: str) -> None:
    console_status("")
    console_status("=== %s ===", title)


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(float(seconds))))
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours:d}h{minutes:02d}m{seconds:02d}s"
    if minutes:
        return f"{minutes:d}m{seconds:02d}s"
    return f"{seconds:d}s"


def format_bytes(byte_count: float) -> str:
    units = ("B", "KB", "MB", "GB", "TB")
    value = float(byte_count)
    for unit in units:
        if value < 1024.0 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024.0
    return f"{value:.2f} {units[-1]}"


def ensure_raw_parser_dependencies() -> None:
    import_module_or_raise("av", package_name="av")
    import_module_or_raise("zstandard", package_name="zstandard")
    import_attr_or_raise("openpilot_logging.logreader", "LogReader", package_name="openpilot-logging")


# =============================================================================
# Training Callbacks
# =============================================================================

class EpochLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch: int, logs: Optional[Dict[str, Any]] = None) -> None:
        values = " | ".join(f"{name}={value:.6f}" for name, value in sorted((logs or {}).items()))
        LOGGER.info("Epoch %d/%d | %s", epoch + 1, EPOCHS, values)


class ValidationRolloutLogger(tf.keras.callbacks.Callback):
    def __init__(self, validation_data: Optional[Dict[str, np.ndarray]] = None, preview_count: int = ROLLOUT_PREVIEW_COUNT) -> None:
        super().__init__()
        self.validation_data = validation_data
        self.preview_count = int(preview_count)
        self.rng = np.random.default_rng(SEED)

    def _select_rollout_window(self) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray]:
        if self.validation_data is None:
            raise RuntimeError("validation data is not available for rollout previews")
        total = int(self.validation_data["telemetry"].shape[0])
        if total == 0:
            raise RuntimeError("validation data is empty")
        count = min(self.preview_count, total)
        max_start = max(total - count, 0)
        start = 0 if max_start == 0 else int(self.rng.integers(0, max_start + 1))
        indices = np.arange(start, start + count, dtype=np.int64)
        inputs = {
            "telemetry": self.validation_data["telemetry"][indices],
            "control_history": self.validation_data["control_history"][indices],
            "image_latent": self.validation_data["image_latent"][indices],
        }
        targets = {
            "throttle_magnitude": self.validation_data["targets"][indices, 0],
            "brake_magnitude": self.validation_data["targets"][indices, 1],
            "steering": self.validation_data["targets"][indices, 2],
        }
        return inputs, targets, indices

    def on_epoch_end(self, epoch: int, logs: Optional[Dict[str, Any]] = None) -> None:
        if (epoch + 1) % int(ROLLOUT_LOG_EVERY_EPOCHS) != 0:
            return
        if self.validation_data is None:
            return
        inputs, targets, indices = self._select_rollout_window()
        predictions = self.model(inputs, training=False)
        console_status(
            "Continuous rollout preview | epoch=%d windows=%d start=%d end=%d",
            epoch + 1,
            len(indices),
            int(indices[0]),
            int(indices[-1]),
        )
        LOGGER.info("Validation rollout preview | epoch=%d", epoch + 1)
        for row_index, sample_index in enumerate(indices):
            LOGGER.info(
                "rollout[%03d] idx=%d | throttle actual=%.4f predicted=%.4f | brake actual=%.4f predicted=%.4f | steering actual=%.4f predicted=%.4f",
                row_index,
                int(sample_index),
                float(targets["throttle_magnitude"][row_index]),
                float(predictions["throttle_magnitude"].numpy().ravel()[row_index]),
                float(targets["brake_magnitude"][row_index]),
                float(predictions["brake_magnitude"].numpy().ravel()[row_index]),
                float(targets["steering"][row_index]),
                float(predictions["steering"].numpy().ravel()[row_index]),
            )


class EpochCheckpoint(tf.keras.callbacks.Callback):
    def __init__(
        self,
        output_dir: Path,
        every_epochs: int = CHECKPOINT_EVERY_EPOCHS,
        seq_len: int = ACTIVE_STACK_SIZE,
        vision_encoder: Optional[tf.keras.Model] = None,
    ) -> None:
        super().__init__()
        self.output_dir = output_dir
        self.every_epochs = int(every_epochs)
        self.seq_len = int(seq_len)
        self.vision_encoder = vision_encoder
        self.last_epoch_number = 0
        self.last_saved_epoch_number = 0

    def _checkpoint_path(self, epoch_number: int) -> Path:
        return self.output_dir / CHECKPOINT_PATTERN.format(epoch=epoch_number)

    def _save_epoch(self, epoch_number: int) -> None:
        path = self._checkpoint_path(epoch_number)
        LOGGER.info("Saving epoch checkpoint %s", path)
        self.model.save(path)
        self.model.save(self.output_dir / "last_checkpoint.keras")
        if EXPORT_POLICY_ONNX:
            try:
                export_and_verify_policy_onnx(path, trained_model=self.model, stack_size=self.seq_len)
            except Exception as exc:
                console_status("Checkpoint ONNX export failed | epoch=%d error=%s", epoch_number, exc)
        if self.vision_encoder is not None:
            vision_keras_path = self.output_dir / f"vision_encoder_epoch_{epoch_number:04d}.keras"
            vision_onnx_path = vision_keras_path.with_suffix(".onnx")
            try:
                self.vision_encoder.save(vision_keras_path)
                console_status("Exported checkpoint vision Keras | epoch=%d path=%s", epoch_number, vision_keras_path)
            except Exception as exc:
                console_status("Checkpoint vision Keras export failed | epoch=%d error=%s", epoch_number, exc)
            if EXPORT_VISION_ONNX:
                try:
                    export_vae_encoder_to_onnx(self.vision_encoder, vision_onnx_path, batch_size=VISION_ONNX_BATCH_SIZE)
                except Exception as exc:
                    console_status("Checkpoint vision ONNX export failed | epoch=%d error=%s", epoch_number, exc)
        console_status("Saved checkpoint | epoch=%d path=%s", epoch_number, path)

    def on_epoch_end(self, epoch: int, logs: Optional[Dict[str, Any]] = None) -> None:
        self.last_epoch_number = epoch + 1
        if self.every_epochs > 0 and (epoch + 1) % self.every_epochs == 0:
            self._save_epoch(epoch + 1)
            self.last_saved_epoch_number = epoch + 1

    def on_train_end(self, logs: Optional[Dict[str, Any]] = None) -> None:
        if self.last_epoch_number and self.last_saved_epoch_number != self.last_epoch_number:
            self._save_epoch(self.last_epoch_number)


# =============================================================================
# CLI Arguments
# =============================================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train a driving policy from drift SDK data")
    parser.add_argument("--dataset", default=os.getenv("DRIFT_DATASET_NAME", "jordy-v0"), help="Dataset name or identifier")
    parser.add_argument("--output-dir", default=os.getenv("DRIFT_OUTPUT_DIR", "outputs"), help="Directory for trained artifacts")
    parser.add_argument("--cache-dir", default=os.getenv("DRIFT_CACHE_DIR", ".cache/drift-data"), help="Local directory for downloaded drift data")
    parser.add_argument("--api-key", default=os.getenv("DRIFT_API_KEY"), help="drift SDK API key")
    parser.add_argument(
        "--dataset-percent",
        type=float,
        default=float(os.getenv("DRIFT_DATASET_PERCENT", str(DATASET_PERCENT))),
        help="Coarse percentage of dataset segments to download and train on (1-100)",
    )
    parser.add_argument("--epochs", type=int, default=EPOCHS, help="Number of training epochs")
    parser.add_argument("--seq-len", type=int, default=STACK_SIZE, help="Sequence length for the recurrent model")
    parser.add_argument("--batch-size", type=int, default=TRAIN_BATCH_SIZE, help="Training batch size")
    parser.add_argument("--lr", type=float, default=LEARNING_RATE, help="Learning rate")
    parser.add_argument("--resume", action="store_true", help="Resume from the latest local checkpoint")
    parser.add_argument("--vae-encoder-path", default=None, help="Path to a local VAE encoder .keras file")
    parser.add_argument("--vae-encoder-export-path", default=None, help="Where to export the resolved VAE encoder locally")
    parser.add_argument("--export-vae-encoder", action="store_true", help="Export the resolved encoder artifact to disk")

    args, unknown = parser.parse_known_args()
    ignored_unknown = []
    for token in unknown:
        if token in {"-f"} or token.startswith("--f=") or token.startswith("--ip=") or token.startswith("--profile"):
            continue
        if token.endswith(".json") and "/jupyter/runtime/" in token:
            continue
        ignored_unknown.append(token)

    if ignored_unknown:
        parser.error(f"unrecognized arguments: {' '.join(ignored_unknown)}")

    if not (0.0 < float(args.dataset_percent) <= 100.0):
        parser.error("--dataset-percent must be > 0 and <= 100")

    return args


# =============================================================================
# Drift SDK Download
# =============================================================================

def _coerce_download_targets(payload: Any) -> List[Any]:
    if payload is None:
        return []
    if isinstance(payload, (list, tuple, set)):
        return [item for item in payload]
    if isinstance(payload, dict):
        for key in ("files", "items", "artifacts", "data", "paths", "entries"):
            value = payload.get(key)
            if isinstance(value, (list, tuple, set)):
                return [item for item in value]
        for key in ("path", "uri", "url", "download_url", "file", "name"):
            if key in payload:
                return [payload]
        return [payload]
    return [payload]


def _materialize_sdk_downloads(candidate: Any, payload: Any, destination: Path) -> Path:
    targets = _coerce_download_targets(payload)
    if not targets:
        return destination

    def download_one(item: Any) -> Optional[Path]:
        if isinstance(item, dict):
            source = item.get("path") or item.get("uri") or item.get("url") or item.get("download_url") or item.get("file") or item.get("name")
            local_name = item.get("name") or item.get("path") or item.get("file") or "artifact"
            if source is None:
                source = item.get("id")
            if source is None:
                return None
        else:
            source = str(item)
            local_name = Path(source).name or "artifact"

        local_path = destination / Path(local_name).name
        local_path.parent.mkdir(parents=True, exist_ok=True)
        for method_name in ("download_file", "download", "get_file"):
            method = getattr(candidate, method_name, None)
            if not callable(method):
                continue
            try:
                method(source, str(local_path))
                return local_path
            except TypeError:
                continue
            except Exception:
                continue
        return None

    with ThreadPoolExecutor(max_workers=max(1, min(4, len(targets)))) as executor:
        futures = [executor.submit(download_one, item) for item in targets]
        for future in as_completed(futures):
            future.result()
    return destination


def _extract_segment_ids(routes: Sequence[Any]) -> List[str]:
    segment_ids: List[str] = []
    for route in routes:
        for segment_ref in getattr(route, "segments", []) or []:
            if isinstance(segment_ref, dict):
                segment_id = segment_ref.get("segment_id")
            else:
                segment_id = getattr(segment_ref, "segment_id", None)
            if segment_id:
                segment_ids.append(str(segment_id))
    return segment_ids


def _download_segments_from_routes(client: Any, dataset_name: str, destination: Path, dataset_percent: float = 100.0) -> Optional[Path]:
    list_routes = getattr(client, "list_routes", None)
    get_route = getattr(client, "get_route", None)
    get_segment = getattr(client, "get_segment", None)
    if not callable(list_routes) or not callable(get_route) or not callable(get_segment):
        return None

    route_summaries = list_routes(dataset_name)
    routes = [get_route(route.route_id) for route in route_summaries]
    segment_ids = _extract_segment_ids(routes)
    original_segment_count = len(segment_ids)
    if segment_ids and float(dataset_percent) < 100.0:
        keep_count = max(1, int(math.ceil(original_segment_count * float(dataset_percent) / 100.0)))
        segment_ids = segment_ids[:keep_count]
    console_status(
        "Drift routes resolved | dataset=%s routes=%d segments=%d selected=%d percent=%.1f",
        dataset_name,
        len(route_summaries),
        original_segment_count,
        len(segment_ids),
        float(dataset_percent),
    )
    if not segment_ids:
        return destination

    def download_segment(segment_id: str) -> Optional[Path]:
        segment = get_segment(segment_id)
        files = getattr(segment, "files", {}) or {}
        segment_dir = destination / segment_id
        segment_dir.mkdir(parents=True, exist_ok=True)
        file_names = {
            "fcamera": "fcamera.hevc",
            "ecamera": "ecamera.hevc",
            "qcamera": "qcamera.ts",
            "qlog": "qlog.zst",
            "rlog": "rlog.zst",
            "navigation": "navigation.json",
        }
        downloaded_any = False
        for alias, filename in file_names.items():
            source_url = files.get(alias)
            if not source_url:
                continue
            target_path = segment_dir / filename
            if target_path.exists() and target_path.stat().st_size > 0:
                downloaded_any = True
                continue
            if alias == "fcamera":
                download = getattr(segment, "download", None)
                if callable(download):
                    download(str(target_path))
                    downloaded_any = True
                    continue
            urllib.request.urlretrieve(source_url, str(target_path))
            downloaded_any = True
        if not downloaded_any:
            return None
        return segment_dir

    with ThreadPoolExecutor(max_workers=max(1, min(4, len(segment_ids)))) as executor:
        futures = [executor.submit(download_segment, segment_id) for segment_id in segment_ids]
        for index, future in enumerate(as_completed(futures), start=1):
            future.result()
            if index == len(segment_ids) or index % 25 == 0:
                console_status(
                    "Drift download progress %d/%d segments (%.1f%%)",
                    index,
                    len(segment_ids),
                    100.0 * float(index) / float(max(len(segment_ids), 1)),
                )
    return destination


def download_dataset_with_sdk(dataset_name: str, destination: Path, api_key: Optional[str] = None, dataset_percent: float = 100.0) -> Path:
    """Download the dataset with the drift SDK instead of using raw S3."""
    destination.mkdir(parents=True, exist_ok=True)

    if any(destination.iterdir()):
        return destination

    client_ctor = getattr(drift, "DriftClient", None) or getattr(drift, "Client", None)
    if client_ctor is None:
        raise RuntimeError("The installed drift SDK does not expose DriftClient")

    try:
        client = client_ctor(api_key=api_key) if api_key else client_ctor()
    except TypeError:
        client = client_ctor()

    try:
        dataset = client.load_dataset(dataset_name)
        console_status(
            "Drift dataset resolved | id=%s routes=%s segments=%s dataset_percent=%.1f",
            dataset.id,
            dataset.routes_count,
            dataset.segments_count,
            float(dataset_percent),
        )
        _download_segments_from_routes(client, dataset_name, destination, dataset_percent=dataset_percent)
    finally:
        close = getattr(client, "close", None)
        if callable(close):
            close()

    downloaded_files = sorted(path for path in destination.glob("**/*") if path.is_file())
    if downloaded_files:
        console_status(
            "Drift download complete | files=%d sample=%s",
            len(downloaded_files),
            downloaded_files[0].name,
        )
        return destination

    raise RuntimeError(
        f"Drift SDK download produced no files in {destination}. dataset={dataset_name}"
    )


# =============================================================================
# Raw Segment Parsing And Alignment
# =============================================================================

def discover_dataset_files(root: Path) -> List[Path]:
    patterns = ["**/*.parquet", "**/*.jsonl", "**/*.json", "**/*.csv"]
    files = []
    for pattern in patterns:
        files.extend(root.glob(pattern))
    return sorted(set(files))


def get_route_name(segment_dir: Path) -> str:
    name, separator, segment_number = segment_dir.name.rpartition("--")
    if separator and segment_number.isdigit():
        return name
    return segment_dir.name


def discover_segments(root: Path) -> List[Tuple[Path, str]]:
    segment_dirs = sorted(
        path.parent
        for path in root.rglob("fcamera.hevc")
        if (path.parent / "rlog.zst").exists()
    )
    if not segment_dirs:
        available = sorted(path.name for path in root.glob("**/*") if path.is_file())
        sample = ", ".join(available[:10]) if available else "no files"
        raise FileNotFoundError(f"No fcamera.hevc/rlog.zst segments found in {root}. Available files: {sample}")
    segments = [(segment_dir, get_route_name(segment_dir)) for segment_dir in segment_dirs]
    console_status(
        "Segment discovery complete | root=%s segments=%d routes=%d",
        root,
        len(segments),
        len({route_name for _, route_name in segments}),
    )
    return segments


def nearest_row(rows: np.ndarray, timestamp_ns: int, max_error_seconds: float) -> Optional[np.ndarray]:
    if len(rows) == 0:
        return None
    times = rows[:, 0]
    index = int(np.searchsorted(times, timestamp_ns))
    candidates = [candidate for candidate in (index - 1, index) if 0 <= candidate < len(rows)]
    best = min(candidates, key=lambda candidate: abs(times[candidate] - timestamp_ns))
    if abs(times[best] - timestamp_ns) > max_error_seconds * 1e9:
        return None
    return rows[best]


def interpolated_row(rows: np.ndarray, timestamp_ns: int, max_gap_seconds: float) -> Optional[np.ndarray]:
    if len(rows) == 0:
        return None
    times = rows[:, 0]
    index = int(np.searchsorted(times, timestamp_ns))
    max_gap_ns = max_gap_seconds * 1e9
    if index < len(rows) and times[index] == timestamp_ns:
        return rows[index]
    if index == 0 or index >= len(rows):
        return None
    before = rows[index - 1]
    after = rows[index]
    before_time = before[0]
    after_time = after[0]
    if timestamp_ns - before_time > max_gap_ns or after_time - timestamp_ns > max_gap_ns:
        return None
    if after_time <= before_time:
        return None
    alpha = (timestamp_ns - before_time) / (after_time - before_time)
    values = before[1:] + alpha * (after[1:] - before[1:])
    return np.concatenate(([float(timestamp_ns)], values))


def stream_rlog(rlog_path: Path):
    zstd = import_module_or_raise("zstandard", package_name="zstandard")
    LogReader = import_attr_or_raise("openpilot_logging.logreader", "LogReader", package_name="openpilot-logging")
    with tempfile.NamedTemporaryFile(prefix="rlog-") as raw_rlog:
        with open(rlog_path, "rb") as compressed:
            with zstd.ZstdDecompressor().stream_reader(compressed) as reader:
                shutil.copyfileobj(reader, raw_rlog)
        raw_rlog.flush()
        yield from LogReader(raw_rlog.name)


def extract_motorola_signal(payload: bytes, start_bit: int, bit_length: int, signed: bool = False) -> int:
    value = 0
    bit = int(start_bit)
    for _ in range(int(bit_length)):
        byte_index = bit // 8
        bit_index = bit % 8
        if byte_index >= len(payload):
            raise ValueError("payload too short for CAN signal")
        value = (value << 1) | ((payload[byte_index] >> bit_index) & 1)
        bit = bit + 15 if bit_index == 0 else bit - 1
    if signed and value & (1 << (bit_length - 1)):
        value -= 1 << bit_length
    return value


def decode_powertrain_control_frame(address: int, payload: bytes) -> Dict[str, float]:
    if address == CAN_ACCELERATOR_PEDAL and len(payload) >= 6:
        return {"throttle": extract_motorola_signal(payload, 47, 8) / 254.0}
    if address == CAN_BRAKE_PEDAL and len(payload) >= 2:
        return {"brake": extract_motorola_signal(payload, 15, 8) / 255.0}
    if address == CAN_STEERING_ANGLE and len(payload) >= 5:
        angle_deg = extract_motorola_signal(payload, 15, 16, signed=True) * 0.0625
        rate_deg_s = extract_motorola_signal(payload, 27, 12, signed=True)
        return {
            "steering": angle_deg / STEERING_ANGLE_SCALE_DEG,
            "steering_rate": rate_deg_s / 720.0,
        }
    return {}


def read_segment_log(rlog_path: Path) -> Dict[str, np.ndarray]:
    camera = []
    controls = []
    telemetry = []
    latest_controls = {"throttle": None, "brake": None, "steering": None, "steering_rate": None}
    latest_control_timestamps = {"throttle": None, "brake": None, "steering": None, "steering_rate": None}
    can_frame_counts = {"throttle": 0, "brake": 0, "steering": 0}

    def signal_is_fresh(signal_name: str, timestamp: int) -> bool:
        signal_timestamp = latest_control_timestamps[signal_name]
        return signal_timestamp is not None and timestamp - signal_timestamp <= CAN_SIGNAL_MAX_AGE_SECONDS * 1e9

    for message in stream_rlog(rlog_path):
        try:
            message_type = message.which()
        except Exception:
            continue
        timestamp = int(message.logMonoTime)
        if message_type == "roadEncodeIdx":
            index = message.roadEncodeIdx
            camera_timestamp = int(index.timestampEof) or timestamp
            camera.append((int(index.segmentId), camera_timestamp))
        elif message_type == "carState":
            state = message.carState
            telemetry.append((timestamp, float(state.vEgo)))
        elif message_type == "can":
            updated_controls = False
            for frame in message.can:
                if int(frame.src) != CAN_BUS_POWERTRAIN:
                    continue
                address = int(frame.address)
                payload = bytes(frame.dat)
                if address == CAN_ACCELERATOR_PEDAL:
                    can_frame_counts["throttle"] += 1
                elif address == CAN_BRAKE_PEDAL:
                    can_frame_counts["brake"] += 1
                elif address == CAN_STEERING_ANGLE:
                    can_frame_counts["steering"] += 1
                decoded = decode_powertrain_control_frame(address, payload)
                for signal_name, value in decoded.items():
                    latest_controls[signal_name] = float(value)
                    latest_control_timestamps[signal_name] = timestamp
                    updated_controls = True
            if not updated_controls:
                continue
            has_fresh_controls = (
                signal_is_fresh("throttle", timestamp)
                and signal_is_fresh("brake", timestamp)
                and signal_is_fresh("steering", timestamp)
                and signal_is_fresh("steering_rate", timestamp)
            )
            if has_fresh_controls:
                controls.append((
                    timestamp,
                    float(np.clip(latest_controls["throttle"], 0.0, 1.0)),
                    float(np.clip(latest_controls["brake"], 0.0, 1.0)),
                    float(np.clip(latest_controls["steering"], -1.0, 1.0)),
                    float(np.clip(latest_controls["steering_rate"], -1.0, 1.0)),
                ))

    def sorted_array(values: List[Tuple[Any, ...]], columns: int) -> np.ndarray:
        if not values:
            return np.empty((0, columns), dtype=np.float64)
        return np.asarray(sorted(values, key=lambda row: row[0]), dtype=np.float64)

    missing = [name for name, count in can_frame_counts.items() if count == 0]
    if missing:
        raise RuntimeError(f"{rlog_path.parent.name} is missing required powertrain CAN signals: {', '.join(missing)}")
    signals = {
        "camera": sorted_array(camera, 2),
        "controls": sorted_array(controls, 5),
        "telemetry": sorted_array(telemetry, 2),
    }
    LOGGER.info(
        "Segment log decoded | segment=%s camera=%d controls=%d telemetry=%d can_frames(throttle=%d brake=%d steering=%d)",
        rlog_path.parent.name,
        len(signals["camera"]),
        len(signals["controls"]),
        len(signals["telemetry"]),
        can_frame_counts["throttle"],
        can_frame_counts["brake"],
        can_frame_counts["steering"],
    )
    return signals


def load_segment(segment_dir: Path) -> Dict[str, np.ndarray]:
    av = import_module_or_raise("av", package_name="av")
    segment_start = time.monotonic()
    console_status("Preprocessing segment | segment=%s", segment_dir.name)
    signals = read_segment_log(segment_dir / "rlog.zst")
    camera_times = {int(frame_index): int(timestamp) for frame_index, timestamp in signals["camera"]}
    frame_stride = max(1, round(SOURCE_VIDEO_FPS / SAMPLE_HZ))
    images = []
    telemetry = []
    control_history = []
    throttle = []
    brake = []
    steering = []

    with av.open(str(segment_dir / "fcamera.hevc"), format="hevc") as video:
        for frame_index, frame in enumerate(video.decode(video=0)):
            if frame_index % frame_stride != 0 or frame_index not in camera_times:
                continue
            timestamp = camera_times[frame_index]
            target_timestamp = timestamp + int(round(CONTROL_TARGET_DELAY_SECONDS * 1e9))
            control = interpolated_row(signals["controls"], target_timestamp, CONTROL_TIMELINE_MAX_GAP_SECONDS)
            current_control = interpolated_row(signals["controls"], timestamp, CONTROL_TIMELINE_MAX_GAP_SECONDS)
            state = nearest_row(signals["telemetry"], timestamp, 0.10)
            if control is None or current_control is None or state is None:
                continue
            if abs(control[0] - target_timestamp) > CONTROL_TARGET_MAX_ERROR_SECONDS * 1e9:
                continue

            image = frame.reformat(width=IMAGE_WIDTH, height=IMAGE_HEIGHT, format="rgb24").to_ndarray()
            images.append(image)
            telemetry.append([np.clip(state[1] / 30.0, 0.0, 2.0)])
            control_history.append([
                np.clip(current_control[1], 0.0, 1.0),
                np.clip(current_control[2], 0.0, 1.0),
                np.clip(current_control[3], -1.0, 1.0),
                np.clip(current_control[4], -1.0, 1.0),
            ])
            throttle.append([float(np.clip(control[1], 0.0, 1.0))])
            brake.append([float(np.clip(control[2], 0.0, 1.0))])
            steering.append([float(np.clip(control[3], -1.0, 1.0))])

    part = {
        "images": np.asarray(images, dtype=np.uint8),
        "telemetry": np.asarray(telemetry, dtype=np.float32),
        "control_history": np.asarray(control_history, dtype=np.float32),
        "throttle": np.asarray(throttle, dtype=np.float32),
        "brake": np.asarray(brake, dtype=np.float32),
        "steering": np.asarray(steering, dtype=np.float32),
    }
    console_status(
        "Segment preprocessing complete | segment=%s aligned_frames=%d elapsed=%s",
        segment_dir.name,
        len(part["images"]),
        format_duration(time.monotonic() - segment_start),
    )
    return part


def load_raw_segment_dataset(dataset_dir: Path) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    ensure_raw_parser_dependencies()
    segments = discover_segments(dataset_dir)
    total_start = time.monotonic()
    parts = []
    sequence_indices = []
    block_ids = []
    offset = 0
    next_block_id = 0
    block_size = max(STACK_SIZE, round(SPLIT_BLOCK_SECONDS * SAMPLE_HZ))
    total_segments = len(segments)

    for segment_number, (segment_dir, route_name) in enumerate(segments, start=1):
        part = load_segment(segment_dir)
        sample_count = len(part["images"])
        if sample_count < STACK_SIZE:
            console_status(
                "Skipping segment | segment=%s route=%s aligned_frames=%d required=%d",
                segment_dir.name,
                route_name,
                sample_count,
                STACK_SIZE,
            )
            console_status(
                "Raw dataset progress %d/%d segments (%.1f%%) kept=%d frames=%d windows=%d elapsed=%s",
                segment_number,
                total_segments,
                100.0 * float(segment_number) / float(max(total_segments, 1)),
                len(parts),
                offset,
                len(sequence_indices),
                format_duration(time.monotonic() - total_start),
            )
            continue
        parts.append(part)
        for block_start in range(0, sample_count, block_size):
            block_end = min(block_start + block_size, sample_count)
            if block_end - block_start < STACK_SIZE:
                continue
            for end in range(block_start + STACK_SIZE - 1, block_end):
                sequence_indices.append(np.arange(end - STACK_SIZE + 1, end + 1) + offset)
                block_ids.append(next_block_id)
            next_block_id += 1
        offset += sample_count
        console_status(
            "Raw dataset progress %d/%d segments (%.1f%%) kept=%d frames=%d windows=%d elapsed=%s",
            segment_number,
            total_segments,
            100.0 * float(segment_number) / float(max(total_segments, 1)),
            len(parts),
            offset,
            len(sequence_indices),
            format_duration(time.monotonic() - total_start),
        )

    if not parts:
        raise RuntimeError("No samples found with aligned camera, vehicle state, and powertrain CAN data.")

    data = {
        key: np.concatenate([part[key] for part in parts], axis=0)
        for key in ("images", "telemetry", "control_history", "throttle", "brake", "steering")
    }
    console_status(
        "Raw dataset materialized | kept_segments=%d frames=%d windows=%d blocks=%d elapsed=%s",
        len(parts),
        len(data["images"]),
        len(sequence_indices),
        len(np.unique(np.asarray(block_ids, dtype=np.int64))),
        format_duration(time.monotonic() - total_start),
    )
    return data, np.asarray(sequence_indices, dtype=np.int64), np.asarray(block_ids, dtype=np.int64)


# =============================================================================
# Tensor Assembly
# =============================================================================

def encode_frame_images(frame: pd.DataFrame, encoder: Optional[tf.keras.Model]) -> np.ndarray:
    image_columns = [column for column in frame.columns if column.lower() in {"image", "images", "rgb_image", "camera_image"}]
    if not image_columns:
        return np.zeros((len(frame), VAE_LATENT_DIM), dtype=np.float32)
    if encoder is None:
        raise ValueError("A local VAE encoder is required when image columns are present")

    image_column = image_columns[0]
    images: List[np.ndarray] = []
    for value in frame[image_column].tolist():
        if value is None:
            raise ValueError("Encountered a missing image value in the dataset")
        if isinstance(value, (bytes, bytearray)):
            array = np.frombuffer(value, dtype=np.uint8)
            if array.size == IMAGE_HEIGHT * IMAGE_WIDTH * 3:
                array = array.reshape(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
            else:
                raise ValueError("Encountered an image buffer with an unexpected size")
        elif isinstance(value, str):
            try:
                decoded = base64.b64decode(value)
                array = np.frombuffer(decoded, dtype=np.uint8)
                if array.size == IMAGE_HEIGHT * IMAGE_WIDTH * 3:
                    array = array.reshape(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
                else:
                    raise ValueError("Encountered a base64 image with an unexpected size")
            except Exception as exc:
                raise ValueError("Failed to decode the image column value") from exc
        elif isinstance(value, np.ndarray):
            array = np.asarray(value, dtype=np.uint8)
            if array.ndim == 1:
                array = array.reshape(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
            elif array.shape != (IMAGE_HEIGHT, IMAGE_WIDTH, 3):
                raise ValueError("Encountered an image array with an unexpected shape")
        else:
            raise ValueError("Unsupported image payload type in the dataset")
        images.append(array.astype(np.uint8))

    batch = np.stack(images, axis=0)
    predictions = encoder(batch, training=False)
    if isinstance(predictions, (list, tuple)):
        latent = predictions[-1]
    else:
        latent = predictions
    return np.asarray(latent, dtype=np.float32)


def make_tensor_data(frame: pd.DataFrame, stack_size: int = STACK_SIZE, latent_dim: int = VAE_LATENT_DIM, image_latents: Optional[np.ndarray] = None) -> Dict[str, np.ndarray]:
    target_candidates = [
        "throttle_magnitude",
        "brake_magnitude",
        "steering",
        "throttle",
        "brake",
        "steering_angle",
        "steering_command",
        "accelerator",
    ]
    target_columns: List[str] = []
    for column in frame.columns:
        if column.lower() in target_candidates or column.lower().startswith("target") or column.lower().startswith("action"):
            target_columns.append(column)

    numeric_columns = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column]) and column not in target_columns]
    feature_columns = numeric_columns

    if feature_columns:
        features = frame[feature_columns].astype(np.float32).to_numpy()
        if features.ndim == 1:
            features = features[:, None]
    else:
        features = np.zeros((len(frame), 1), dtype=np.float32)

    if features.shape[0] < stack_size:
        raise ValueError("Dataset is too short for the requested stack size")

    if features.shape[1] == 0:
        features = np.zeros((len(frame), 1), dtype=np.float32)

    if features.shape[1] < 1:
        features = np.pad(features, ((0, 0), (0, 1 - features.shape[1])), mode="constant")

    telemetry = features[:, :1]
    control_history = features[:, 1:5]
    if control_history.shape[1] < CONTROL_HISTORY_DIM:
        control_history = np.pad(control_history, ((0, 0), (0, CONTROL_HISTORY_DIM - control_history.shape[1])), mode="constant")

    if image_latents is None:
        raise ValueError("Image latents must be provided by the VAE encoder")

    image_latent = np.asarray(image_latents, dtype=np.float32)
    if image_latent.ndim == 1:
        image_latent = image_latent[:, None]
    if image_latent.shape[0] != len(frame):
        raise ValueError("The VAE encoder output length does not match the dataset length")
    if image_latent.shape[1] < latent_dim:
        image_latent = np.pad(image_latent, ((0, 0), (0, latent_dim - image_latent.shape[1])), mode="constant")
    image_latent = image_latent[:, :latent_dim].astype(np.float32)

    if target_columns:
        targets = frame[target_columns].astype(np.float32).to_numpy()
        if targets.ndim == 1:
            targets = targets[:, None]
    else:
        targets = np.column_stack(
            [
                np.tanh(telemetry[:, 0]),
                np.tanh(np.mean(control_history[:, :2], axis=1)),
                np.sin(np.linspace(0.0, 2.0 * np.pi, len(frame))),
            ]
        ).astype(np.float32)

    if targets.shape[1] < 3:
        target_pad = np.zeros((len(frame), 3 - targets.shape[1]), dtype=np.float32)
        targets = np.concatenate([targets, target_pad], axis=1)

    targets = targets[:, :3]
    target_names = ["throttle_magnitude", "brake_magnitude", "steering"]

    sequence_indices = np.arange(len(frame) - stack_size + 1)
    sequence_windows = [np.arange(index, index + stack_size) for index in sequence_indices]

    telemetry_sequences = np.stack([telemetry[idx] for idx in sequence_windows], axis=0)
    control_sequences = np.stack([control_history[idx] for idx in sequence_windows], axis=0)
    image_sequences = np.stack([image_latent[idx] for idx in sequence_windows], axis=0)
    target_sequences = np.stack([targets[idx[-1]] for idx in sequence_windows], axis=0)

    return {
        "telemetry": telemetry_sequences.astype(np.float32),
        "control_history": control_sequences.astype(np.float32),
        "image_latent": image_sequences.astype(np.float32),
        "targets": target_sequences.astype(np.float32),
        "target_names": target_names,
    }


def make_tensor_data_from_raw(data: Dict[str, np.ndarray], sequence_indices: np.ndarray, latent_dim: int, image_latents: np.ndarray) -> Dict[str, np.ndarray]:
    final_indices = sequence_indices[:, -1]
    return {
        "telemetry": data["telemetry"][sequence_indices].astype(np.float32),
        "control_history": data["control_history"][sequence_indices].astype(np.float32),
        "image_latent": image_latents[sequence_indices].astype(np.float32),
        "targets": np.concatenate(
            [
                data["throttle"][final_indices],
                data["brake"][final_indices],
                data["steering"][final_indices],
            ],
            axis=1,
        ).astype(np.float32),
        "target_names": ["throttle_magnitude", "brake_magnitude", "steering"],
    }


def split_tensor_data(tensor_data: Dict[str, np.ndarray], val_ratio: float = VALIDATION_FRACTION, seed: int = SEED) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    count = tensor_data["telemetry"].shape[0]
    if count < 2:
        return tensor_data, tensor_data
    split_idx = max(1, int(count * (1.0 - val_ratio)))
    rng = np.random.default_rng(seed)
    indices = np.arange(count)
    rng.shuffle(indices)
    train_idx = indices[:split_idx]
    val_idx = indices[split_idx:]

    def split_mapping(mapping: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        return {key: value[train_idx if key == "targets" else train_idx] for key, value in mapping.items()}  # type: ignore[misc]

    train_data = {key: value[train_idx] for key, value in tensor_data.items() if key != "target_names"}
    val_data = {key: value[val_idx] for key, value in tensor_data.items() if key != "target_names"}
    return train_data, val_data


def make_tf_dataset(tensor_data: Dict[str, np.ndarray], indices: np.ndarray, training: bool, batch_size: int = TRAIN_BATCH_SIZE) -> tf.data.Dataset:
    telemetry = tf.convert_to_tensor(tensor_data["telemetry"], dtype=tf.float32)
    control_history = tf.convert_to_tensor(tensor_data["control_history"], dtype=tf.float32)
    image_latent = tf.convert_to_tensor(tensor_data["image_latent"], dtype=tf.float32)
    targets = tf.convert_to_tensor(tensor_data["targets"], dtype=tf.float32)

    sample_count = telemetry.shape[0]
    if sample_count == 0:
        raise ValueError("No training sequences are available")

    dataset = tf.data.Dataset.from_tensor_slices((telemetry, control_history, image_latent, targets))
    if training:
        dataset = dataset.shuffle(min(sample_count, 10_000), seed=SEED)

    def map_sequence(telemetry_sequence: tf.Tensor, control_sequence: tf.Tensor, image_sequence: tf.Tensor, target: tf.Tensor) -> Tuple[Dict[str, tf.Tensor], Dict[str, tf.Tensor]]:
        inputs = {
            "telemetry": telemetry_sequence,
            "control_history": control_sequence,
            "image_latent": image_sequence,
        }
        targets_dict = {
            "throttle_magnitude": tf.expand_dims(target[0:1], axis=0),
            "brake_magnitude": tf.expand_dims(target[1:2], axis=0),
            "steering": tf.expand_dims(target[2:3], axis=0),
        }
        return inputs, targets_dict

    return dataset.map(map_sequence, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)


# =============================================================================
# Model Building And Export
# =============================================================================

@tf.keras.utils.register_keras_serializable()
class Sampling(tf.keras.layers.Layer):
    def call(self, inputs: Tuple[tf.Tensor, tf.Tensor]) -> tf.Tensor:
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


def residual_block(x: tf.Tensor, filters: int, name: str) -> tf.Tensor:
    shortcut = x
    if int(x.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, padding="same", name=f"{name}_skip")(shortcut)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="swish", name=f"{name}_conv1")(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", name=f"{name}_conv2")(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_bn2")(x)
    x = tf.keras.layers.Add(name=f"{name}_add")([shortcut, x])
    return tf.keras.layers.Activation("swish", name=f"{name}_out")(x)


def build_vae_encoder(image_height: int = IMAGE_HEIGHT, image_width: int = IMAGE_WIDTH) -> tf.keras.Model:
    image_input = tf.keras.Input((image_height, image_width, 3), dtype=tf.uint8, name="image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="image_rescaling")(image_input)
    for block_index, filters in enumerate((32, 64, 128, 192), start=1):
        x = tf.keras.layers.SeparableConv2D(
            filters,
            3,
            strides=2,
            padding="same",
            activation="swish",
            name=f"encoder_down{block_index}_sepconv",
        )(x)
        x = tf.keras.layers.BatchNormalization(name=f"encoder_down{block_index}_bn")(x)
        x = residual_block(x, filters, name=f"encoder_down{block_index}_res")
    x = residual_block(x, 256, name="encoder_bottleneck_res1")
    x = residual_block(x, 256, name="encoder_bottleneck_res2")
    x = tf.keras.layers.Conv2D(LATENT_CHANNELS, 1, padding="same", name="z_conv")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name="z_pool")(x)
    z_mean = tf.keras.layers.Dense(VAE_LATENT_DIM, activation=None, name="z_mean")(x)
    z_log_var = tf.keras.layers.Dense(VAE_LATENT_DIM, activation=None, name="z_log_var")(x)
    z = Sampling(name="z")([z_mean, z_log_var])
    return tf.keras.Model(image_input, [z_mean, z_log_var, z], name="irl_vae_encoder")


def load_local_vae_encoder(path: Optional[str | os.PathLike[str]]) -> tf.keras.Model:
    if not path:
        raise ValueError("No encoder path was provided")
    encoder_path = Path(path)
    if not encoder_path.exists():
        raise FileNotFoundError(f"VAE encoder path does not exist: {encoder_path}")
    return tf.keras.models.load_model(
        str(encoder_path),
        custom_objects={"Sampling": Sampling},
        compile=False,
        safe_mode=False,
    )


def export_local_vae_encoder(encoder: tf.keras.Model, output_path: Optional[str | os.PathLike[str]]) -> Path:
    if not output_path:
        raise ValueError("No export path was provided")
    export_path = Path(output_path)
    export_path.parent.mkdir(parents=True, exist_ok=True)
    encoder.save(str(export_path))
    return export_path


def export_vae_encoder_to_onnx(encoder: tf.keras.Model, output_path: str | os.PathLike[str], batch_size: int = VISION_ONNX_BATCH_SIZE) -> Path:
    tf2onnx = import_module_or_raise("tf2onnx", package_name="tf2onnx")
    export_path = Path(output_path)
    export_path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(encoder.output, (list, tuple)):
        export_model = tf.keras.Model(encoder.input, encoder.output[-1], name=f"{encoder.name}_export")
    else:
        export_model = encoder
    spec = (tf.TensorSpec([batch_size, IMAGE_HEIGHT, IMAGE_WIDTH, 3], tf.uint8, name="image"),)
    _ = tf2onnx.convert.from_keras(export_model, input_signature=spec, opset=POLICY_ONNX_OPSET, output_path=str(export_path))
    console_status("Exported vision ONNX | onnx=%s", export_path)
    return export_path


def policy_onnx_report(onnx_path: str | os.PathLike[str]) -> Dict[str, Any]:
    onnx = import_module_or_raise("onnx", package_name="onnx")
    model = onnx.load(str(onnx_path))
    op_counts: Dict[str, int] = {}
    for node in model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

    def value_info(value: Any) -> Dict[str, Any]:
        shape = [dim.dim_value or dim.dim_param or None for dim in value.type.tensor_type.shape.dim]
        return {"name": value.name, "shape": shape}

    return {
        "path": str(onnx_path),
        "inputs": [value_info(value) for value in model.graph.input],
        "outputs": [value_info(value) for value in model.graph.output],
        "op_counts": dict(sorted(op_counts.items())),
        "disallowed_ops": sorted(op for op in op_counts if op in POLICY_ONNX_DISALLOWED_OPS),
    }


def verify_policy_onnx_for_tinygrad(onnx_path: str | os.PathLike[str], stack_size: int = ACTIVE_STACK_SIZE) -> Dict[str, Any]:
    report = policy_onnx_report(onnx_path)
    if report["disallowed_ops"]:
        raise RuntimeError("Policy ONNX contains Tinygrad-unsupported ops: " + ", ".join(report["disallowed_ops"]))
    if POLICY_ONNX_SINGLE_OUTPUT:
        output_names = [output["name"] for output in report["outputs"]]
        if output_names != [POLICY_ONNX_SINGLE_OUTPUT_NAME]:
            raise RuntimeError(
                "Policy ONNX output contract mismatch: "
                f"expected {[POLICY_ONNX_SINGLE_OUTPUT_NAME]}, got {output_names}"
            )
        output_shape = report["outputs"][0]["shape"]
        expected_shape = [POLICY_ONNX_BATCH_SIZE, len(POLICY_ONNX_OUTPUT_NAMES)]
        if output_shape != expected_shape:
            raise RuntimeError(f"Policy ONNX output shape mismatch: expected {expected_shape}, got {output_shape}")

    if VERIFY_POLICY_ONNX_TINYGRAD:
        OnnxRunner = import_attr_or_raise("tinygrad.nn.onnx", "OnnxRunner", package_name="tinygrad")
        runner = OnnxRunner(str(onnx_path))
        inputs = {
            "image_latent": np.zeros((POLICY_ONNX_BATCH_SIZE, stack_size, VAE_LATENT_DIM), dtype=np.float32),
            "telemetry": np.zeros((POLICY_ONNX_BATCH_SIZE, stack_size, 1), dtype=np.float32),
            "control_history": np.zeros((POLICY_ONNX_BATCH_SIZE, stack_size, CONTROL_HISTORY_DIM), dtype=np.float32),
        }
        outputs = runner(inputs)
        tinygrad_outputs: Dict[str, Dict[str, Any]] = {}
        for name, value in outputs.items():
            if hasattr(value, "contiguous"):
                value = value.contiguous().realize()
            array = value.numpy() if hasattr(value, "numpy") else np.asarray(value)
            tinygrad_outputs[name] = {"shape": list(array.shape), "dtype": str(array.dtype)}
        report["tinygrad_outputs"] = tinygrad_outputs

    report_path = Path(onnx_path).with_suffix(".onnx.json")
    report_path.write_text(json.dumps(report, indent=2))
    console_status("Verified policy ONNX | path=%s report=%s", onnx_path, report_path)
    return report


def export_and_verify_policy_onnx(
    keras_model_path: str | os.PathLike[str],
    *,
    trained_model: Optional[tf.keras.Model] = None,
    batch_size: int = POLICY_ONNX_BATCH_SIZE,
    stack_size: int = ACTIVE_STACK_SIZE,
) -> Optional[Path]:
    if not EXPORT_POLICY_ONNX:
        return None
    keras_model_path = Path(keras_model_path)
    onnx_path = keras_model_path.with_suffix(".onnx")
    export_irl_policy_to_onnx(
        keras_model_path,
        onnx_path,
        trained_model=trained_model,
        batch_size=batch_size,
        stack_size=stack_size,
    )
    verify_policy_onnx_for_tinygrad(onnx_path, stack_size=stack_size)
    return onnx_path


def resolve_vae_encoder(vae_encoder_path: Optional[str | os.PathLike[str]], export_path: Optional[str | os.PathLike[str]], export_enabled: bool) -> tf.keras.Model:
    if vae_encoder_path:
        encoder = load_local_vae_encoder(vae_encoder_path)
        if export_enabled and export_path:
            export_local_vae_encoder(encoder, export_path)
        return encoder

    encoder = build_vae_encoder()
    if export_enabled and export_path:
        export_local_vae_encoder(encoder, export_path)
    return encoder


def _policy_outputs_for_export(outputs: Dict[str, tf.Tensor]) -> tf.Tensor | List[tf.Tensor]:
    if POLICY_ONNX_SINGLE_OUTPUT:
        return tf.keras.layers.Concatenate(axis=-1, name=POLICY_ONNX_SINGLE_OUTPUT_NAME)([outputs[name] for name in POLICY_ONNX_OUTPUT_NAMES])
    return [tf.keras.layers.Lambda(lambda value: value, name=name)(outputs[name]) for name in POLICY_ONNX_OUTPUT_NAMES]


def _weighted_dense_from_layer(source_layer: tf.keras.layers.Layer, name: str) -> tf.keras.layers.Dense:
    return tf.keras.layers.Dense(
        source_layer.units,
        activation=source_layer.activation,
        use_bias=source_layer.use_bias,
        name=name,
    )


def _build_standard_policy_export_model(trained_model: tf.keras.Model, *, batch_size: int, stack_size: int) -> tf.keras.Model:
    image_latent_in = tf.keras.Input(shape=(stack_size, VAE_LATENT_DIM), batch_size=batch_size, dtype=tf.float32, name="image_latent")
    telemetry_in = tf.keras.Input(shape=(stack_size, 1), batch_size=batch_size, dtype=tf.float32, name="telemetry")
    control_history_in = tf.keras.Input(shape=(stack_size, CONTROL_HISTORY_DIM), batch_size=batch_size, dtype=tf.float32, name="control_history")

    outputs = trained_model(
        {
            "image_latent": image_latent_in,
            "telemetry": telemetry_in,
            "control_history": control_history_in,
        },
        training=False,
    )
    return tf.keras.Model(
        inputs=[image_latent_in, telemetry_in, control_history_in],
        outputs=_policy_outputs_for_export(outputs),
        name="irl_policy_bolt_export",
    )


def _build_fast_policy_export_model(trained_model: tf.keras.Model, *, batch_size: int, stack_size: int) -> tf.keras.Model:
    image_latent_in = tf.keras.Input(shape=(stack_size, VAE_LATENT_DIM), batch_size=batch_size, dtype=tf.float32, name="image_latent")
    telemetry_in = tf.keras.Input(shape=(stack_size, 1), batch_size=batch_size, dtype=tf.float32, name="telemetry")
    control_history_in = tf.keras.Input(shape=(stack_size, CONTROL_HISTORY_DIM), batch_size=batch_size, dtype=tf.float32, name="control_history")

    def time_distributed_dense(name: str, inputs: tf.Tensor) -> tf.Tensor:
        source = trained_model.get_layer(name)
        inner_layer = getattr(source, "layer", source)
        layer = _weighted_dense_from_layer(inner_layer, name)
        outputs = layer(inputs)
        layer.set_weights(inner_layer.get_weights())
        return outputs

    image_latent_tokens = time_distributed_dense("image_latent_projection", image_latent_in)
    telemetry_tokens = time_distributed_dense("telemetry_projection", telemetry_in)
    control_tokens = time_distributed_dense("control_history_projection", control_history_in)
    per_timestep_features = tf.keras.layers.Concatenate(axis=-1, name="per_timestep_fusion")([image_latent_tokens, telemetry_tokens, control_tokens])
    per_timestep_features = time_distributed_dense("per_timestep_projection", per_timestep_features)

    temporal_source = trained_model.get_layer("temporal_simple_rnn")
    input_kernel, recurrent_kernel, bias = temporal_source.get_weights()
    input_projection = tf.keras.layers.Dense(temporal_source.units, activation=None, use_bias=True, name="temporal_simple_rnn_input_projection")
    recurrent_projection = tf.keras.layers.Dense(temporal_source.units, activation=None, use_bias=False, name="temporal_simple_rnn_recurrent_projection")

    initial_step = tf.keras.layers.Lambda(lambda tensor: tensor[:, 0, :], name="temporal_simple_rnn_step_00")(per_timestep_features)
    state = input_projection(initial_step)
    state = tf.keras.layers.Activation(temporal_source.activation, name="temporal_simple_rnn_state_00")(state)
    _ = recurrent_projection(state)
    input_projection.set_weights([input_kernel, bias])
    recurrent_projection.set_weights([recurrent_kernel])

    for step in range(1, int(stack_size)):
        current_step = tf.keras.layers.Lambda(lambda tensor, index=step: tensor[:, index, :], name=f"temporal_simple_rnn_step_{step:02d}")(per_timestep_features)
        projected_input = input_projection(current_step)
        projected_state = recurrent_projection(state)
        state = tf.keras.layers.Add(name=f"temporal_simple_rnn_add_{step:02d}")([projected_input, projected_state])
        state = tf.keras.layers.Activation(temporal_source.activation, name=f"temporal_simple_rnn_state_{step:02d}")(state)

    features = state
    for name in ("policy_dense_1", "policy_dense_2"):
        source = trained_model.get_layer(name)
        layer = _weighted_dense_from_layer(source, name)
        features = layer(features)
        layer.set_weights(source.get_weights())

    outputs = {}
    for name in POLICY_ONNX_OUTPUT_NAMES:
        source = trained_model.get_layer(name)
        layer = _weighted_dense_from_layer(source, name)
        outputs[name] = layer(features)
        layer.set_weights(source.get_weights())

    return tf.keras.Model(
        inputs=[image_latent_in, telemetry_in, control_history_in],
        outputs=_policy_outputs_for_export(outputs),
        name="irl_policy_bolt_fast_export",
    )


def export_irl_policy_to_onnx(keras_model_path: str | os.PathLike[str], onnx_path: str | os.PathLike[str], *, trained_model: Optional[tf.keras.Model] = None, batch_size: int = POLICY_ONNX_BATCH_SIZE, stack_size: int = ACTIVE_STACK_SIZE) -> Path:
    tf2onnx = import_module_or_raise("tf2onnx", package_name="tf2onnx")
    keras_path = Path(keras_model_path)
    onnx_output_path = Path(onnx_path)
    onnx_output_path.parent.mkdir(parents=True, exist_ok=True)

    if trained_model is None:
        trained_model = tf.keras.models.load_model(str(keras_path), compile=False, safe_mode=False)

    export_model = _build_fast_policy_export_model(trained_model, batch_size=batch_size, stack_size=stack_size) if POLICY_ONNX_FAST_RANK3_DENSE else _build_standard_policy_export_model(trained_model, batch_size=batch_size, stack_size=stack_size)

    spec = (
        tf.TensorSpec([batch_size, stack_size, VAE_LATENT_DIM], tf.float32, name="image_latent"),
        tf.TensorSpec([batch_size, stack_size, 1], tf.float32, name="telemetry"),
        tf.TensorSpec([batch_size, stack_size, CONTROL_HISTORY_DIM], tf.float32, name="control_history"),
    )

    _ = tf2onnx.convert.from_keras(export_model, input_signature=spec, opset=POLICY_ONNX_OPSET, output_path=str(onnx_output_path))
    console_status("Exported ONNX | keras=%s onnx=%s fast_rank3_dense=%s single_output=%s", keras_path, onnx_output_path, POLICY_ONNX_FAST_RANK3_DENSE, POLICY_ONNX_SINGLE_OUTPUT)
    return onnx_output_path


def build_model() -> tf.keras.Model:
    telemetry_input = tf.keras.Input(shape=(ACTIVE_STACK_SIZE, 1), name="telemetry")
    control_history_input = tf.keras.Input(shape=(ACTIVE_STACK_SIZE, CONTROL_HISTORY_DIM), name="control_history")
    image_latent_input = tf.keras.Input(shape=(ACTIVE_STACK_SIZE, VAE_LATENT_DIM), name="image_latent")

    telemetry_tokens = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(32, activation="swish"), name="telemetry_projection")(telemetry_input)
    control_tokens = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(32, activation="swish"), name="control_history_projection")(control_history_input)
    image_tokens = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(128, activation="swish"), name="image_latent_projection")(image_latent_input)
    per_timestep_features = tf.keras.layers.Concatenate(axis=-1, name="per_timestep_fusion")([telemetry_tokens, control_tokens, image_tokens])
    per_timestep_features = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(256, activation="swish"), name="per_timestep_projection")(per_timestep_features)
    features = tf.keras.layers.SimpleRNN(256, activation="tanh", name="temporal_simple_rnn")(per_timestep_features)
    features = tf.keras.layers.Dense(256, activation="swish", name="policy_dense_1")(features)
    features = tf.keras.layers.Dropout(0.10)(features)
    features = tf.keras.layers.Dense(128, activation="swish", name="policy_dense_2")(features)

    throttle_magnitude = tf.keras.layers.Dense(1, activation="sigmoid", name="throttle_magnitude")(features)
    brake_magnitude = tf.keras.layers.Dense(1, activation="sigmoid", name="brake_magnitude")(features)
    steering = tf.keras.layers.Dense(1, activation="tanh", name="steering")(features)

    model = tf.keras.Model(
        inputs={"telemetry": telemetry_input, "control_history": control_history_input, "image_latent": image_latent_input},
        outputs={"throttle_magnitude": throttle_magnitude, "brake_magnitude": brake_magnitude, "steering": steering},
        name="irl_policy_bolt",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={
            "throttle_magnitude": tf.keras.losses.MeanSquaredError(),
            "brake_magnitude": tf.keras.losses.MeanSquaredError(),
            "steering": tf.keras.losses.MeanSquaredError(),
        },
        loss_weights={
            "throttle_magnitude": THROTTLE_LOSS_WEIGHT,
            "brake_magnitude": BRAKE_LOSS_WEIGHT,
            "steering": STEERING_LOSS_WEIGHT,
        },
    )
    return model


# =============================================================================
# Training State And Artifact Persistence
# =============================================================================

def save_training_state(output_dir: Path, state: Dict[str, Any]) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "training_state.json").open("w", encoding="utf-8") as handle:
        json.dump(state, handle, indent=2)


def load_training_state(output_dir: Path) -> Optional[Dict[str, Any]]:
    state_path = output_dir / "training_state.json"
    if not state_path.exists():
        return None
    with state_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_metrics(output_dir: Path, step: int, train_loss: float, val_loss: float) -> None:
    progress_path = output_dir / "progress_log.jsonl"
    with progress_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps({"step": step, "train_loss": train_loss, "val_loss": val_loss}) + "\n")
    history_csv = output_dir / "epoch_metrics.csv"
    if not history_csv.exists():
        with history_csv.open("w", encoding="utf-8") as handle:
            handle.write("step,train_loss,val_loss\n")
    with history_csv.open("a", encoding="utf-8") as handle:
        handle.write(f"{step},{train_loss},{val_loss}\n")


def checkpoint_path(output_dir: Path, epoch: int) -> Path:
    return output_dir / CHECKPOINT_PATTERN.format(epoch=epoch)


def latest_checkpoint(output_dir: Path) -> Optional[Path]:
    checkpoints = sorted(output_dir.glob("policy_epoch_*.keras"))
    return checkpoints[-1] if checkpoints else None


# =============================================================================
# Training Loop
# =============================================================================

def train_model(model: tf.keras.Model, train_dataset: tf.data.Dataset, val_dataset: tf.data.Dataset, output_dir: Path, args: argparse.Namespace) -> Dict[str, Any]:
    history = {"epochs": 0, "best_val_loss": float("inf"), "train_loss": [], "val_loss": []}
    state = load_training_state(output_dir)
    if state is not None and args.resume:
        history = state.get("history", history)
        history["epochs"] = int(state.get("history", {}).get("epochs", 0))
        history["best_val_loss"] = float(state.get("history", {}).get("best_val_loss", float("inf")))
        history["train_loss"] = list(state.get("history", {}).get("train_loss", []))
        history["val_loss"] = list(state.get("history", {}).get("val_loss", []))
        ckpt_path = latest_checkpoint(output_dir)
        if ckpt_path is not None:
            model = tf.keras.models.load_model(ckpt_path, compile=False)
            console_status("Resumed from checkpoint | path=%s", ckpt_path)

    callbacks = [
        EpochLogger(),
        EpochCheckpoint(output_dir, every_epochs=CHECKPOINT_EVERY_EPOCHS, seq_len=args.seq_len),
    ]
    console_status(
        "Training start | epochs=%d train_batches=%s val_batches=%s output_dir=%s",
        args.epochs,
        tf.data.experimental.cardinality(train_dataset).numpy(),
        tf.data.experimental.cardinality(val_dataset).numpy(),
        output_dir,
    )

    history_obj = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=args.epochs,
        callbacks=callbacks,
        verbose=0,
    )

    metrics = {
        "epochs": len(history_obj.history.get("loss", [])),
        "best_val_loss": float(np.min(history_obj.history.get("val_loss", [float("inf")]))),
        "train_loss": list(history_obj.history.get("loss", [])),
        "val_loss": list(history_obj.history.get("val_loss", [])),
    }
    save_training_state(output_dir, {"history": metrics})
    console_status(
        "Training complete | epochs=%d best_val_loss=%.6f",
        metrics["epochs"],
        metrics["best_val_loss"],
    )
    return metrics


def save_policy_artifacts(output_dir: Path, model: tf.keras.Model, feature_names: Sequence[str], target_names: Sequence[str], history: Dict[str, Any], seq_len: int) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    final_epoch = int(history.get("epochs", 0))
    keras_path = output_dir / f"final_policy_epoch_{final_epoch:04d}.keras"
    model.save(keras_path)
    metadata = {
        "feature_names": list(feature_names),
        "target_names": list(target_names),
        "model_type": "rnn_behavior_clone",
        "history": history,
    }
    with (output_dir / "policy_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    try:
        export_and_verify_policy_onnx(keras_path, trained_model=model, batch_size=POLICY_ONNX_BATCH_SIZE, stack_size=seq_len)
    except Exception as exc:
        console_status("Final policy ONNX export failed | error=%s", exc)


def save_vision_artifacts(output_dir: Path, vae_encoder: tf.keras.Model, encoder_export_path: Path, history: Dict[str, Any]) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    final_epoch = int(history.get("epochs", 0))
    keras_path = output_dir / f"vision_encoder_epoch_{final_epoch:04d}.keras"
    onnx_path = keras_path.with_suffix(".onnx")
    vae_encoder.save(keras_path)
    console_status("Exported vision Keras | path=%s", keras_path)
    if EXPORT_VISION_ONNX:
        try:
            export_vae_encoder_to_onnx(vae_encoder, onnx_path, batch_size=VISION_ONNX_BATCH_SIZE)
        except Exception as exc:
            console_status("Vision ONNX export failed | error=%s", exc)


# =============================================================================
# Main Entry Point
# =============================================================================

def main() -> None:
    args = parse_args()
    dataset_name = args.dataset
    output_dir = Path(args.output_dir)
    cache_dir = Path(args.cache_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    export_path = Path(args.vae_encoder_export_path) if args.vae_encoder_export_path else output_dir / "encoder.keras"
    vae_encoder = resolve_vae_encoder(args.vae_encoder_path, export_path, args.export_vae_encoder)
    log_section("Setup")
    console_status("Resolved VAE encoder | path=%s", export_path)

    log_section("Download")
    console_status(
        "Downloading dataset via drift SDK | dataset=%s cache_dir=%s dataset_percent=%.1f",
        dataset_name,
        cache_dir,
        float(args.dataset_percent),
    )
    download_dataset_with_sdk(
        dataset_name=dataset_name,
        destination=cache_dir,
        api_key=args.api_key,
        dataset_percent=float(args.dataset_percent),
    )

    log_section("Preprocess")
    console_status("Loading raw segment dataset | cache_dir=%s", cache_dir)
    raw_data, sequence_indices, block_ids = load_raw_segment_dataset(cache_dir)
    console_status(
        "Loaded raw dataset | image_shape=%s telemetry_shape=%s control_shape=%s windows=%d",
        raw_data["images"].shape,
        raw_data["telemetry"].shape,
        raw_data["control_history"].shape,
        len(sequence_indices),
    )

    log_section("Encode")
    image_latents = encode_frame_images(pd.DataFrame({"image": list(raw_data["images"])}), vae_encoder)
    console_status(
        "Image encoding complete | frames=%d latent_shape=%s",
        len(raw_data["images"]),
        image_latents.shape,
    )
    tensor_data = make_tensor_data_from_raw(raw_data, sequence_indices, latent_dim=VAE_LATENT_DIM, image_latents=image_latents)
    console_status(
        "Tensor assembly complete | telemetry=%s control_history=%s image_latent=%s targets=%s",
        tensor_data["telemetry"].shape,
        tensor_data["control_history"].shape,
        tensor_data["image_latent"].shape,
        tensor_data["targets"].shape,
    )

    log_section("Split")
    train_data, val_data = split_tensor_data(tensor_data, val_ratio=VALIDATION_FRACTION, seed=SEED)
    console_status(
        "Dataset split complete | train_windows=%d val_windows=%d val_fraction=%.2f",
        train_data["telemetry"].shape[0],
        val_data["telemetry"].shape[0],
        VALIDATION_FRACTION,
    )
    train_dataset = make_tf_dataset(train_data, np.arange(train_data["telemetry"].shape[0]), training=True, batch_size=args.batch_size)
    val_dataset = make_tf_dataset(val_data, np.arange(val_data["telemetry"].shape[0]), training=False, batch_size=args.batch_size)

    log_section("Train")
    model = build_model()
    rollout_logger = ValidationRolloutLogger(validation_data=val_data)
    def train_model_with_rollout(model: tf.keras.Model, train_dataset: tf.data.Dataset, val_dataset: tf.data.Dataset, output_dir: Path, args: argparse.Namespace) -> Dict[str, Any]:
        history = {"epochs": 0, "best_val_loss": float("inf"), "train_loss": [], "val_loss": []}
        state = load_training_state(output_dir)
        if state is not None and args.resume:
            history = state.get("history", history)
            history["epochs"] = int(state.get("history", {}).get("epochs", 0))
            history["best_val_loss"] = float(state.get("history", {}).get("best_val_loss", float("inf")))
            history["train_loss"] = list(state.get("history", {}).get("train_loss", []))
            history["val_loss"] = list(state.get("history", {}).get("val_loss", []))
            ckpt_path = latest_checkpoint(output_dir)
            if ckpt_path is not None:
                model = tf.keras.models.load_model(ckpt_path, compile=False)
                console_status("Resumed from checkpoint | path=%s", ckpt_path)

        callbacks = [
            EpochLogger(),
            EpochCheckpoint(
                output_dir,
                every_epochs=CHECKPOINT_EVERY_EPOCHS,
                seq_len=args.seq_len,
                vision_encoder=vae_encoder,
            ),
            rollout_logger,
        ]
        console_status(
            "Training start | epochs=%d train_batches=%s val_batches=%s output_dir=%s",
            args.epochs,
            tf.data.experimental.cardinality(train_dataset).numpy(),
            tf.data.experimental.cardinality(val_dataset).numpy(),
            output_dir,
        )
        history_obj = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=args.epochs,
            callbacks=callbacks,
            verbose=0,
        )
        metrics = {
            "epochs": len(history_obj.history.get("loss", [])),
            "best_val_loss": float(np.min(history_obj.history.get("val_loss", [float("inf")]))),
            "train_loss": list(history_obj.history.get("loss", [])),
            "val_loss": list(history_obj.history.get("val_loss", [])),
        }
        save_training_state(output_dir, {"history": metrics})
        console_status(
            "Training complete | epochs=%d best_val_loss=%.6f",
            metrics["epochs"],
            metrics["best_val_loss"],
        )
        return metrics

    history = train_model_with_rollout(model, train_dataset, val_dataset, output_dir, args)

    log_section("Export")
    save_policy_artifacts(
        output_dir,
        model,
        ["images", "telemetry", "control_history"],
        list(tensor_data["target_names"]),
        history,
        args.seq_len,
    )
    save_vision_artifacts(output_dir, vae_encoder, export_path, history)
    console_status("Run complete | output_dir=%s", output_dir)


if __name__ == "__main__":
    main()
